# Janus S1 Fine-Tune — intent head on Kaggle 2×T4

Fine-tunes a Qwen3-0.6B-class model on the janus decision corpus
(3,149 labeled decisions: prompt → gold_intent) to produce a sharper
System 1 router. Free Kaggle GPU, ~30-60 min.

**Guidance-weighting (GA-GRPO λ\*)**: the corpus labels are the guidance
signal — weight them heaviest early, decay as the model converges
(class weights below implement the imbalance half of that).

**Class imbalance**: 91% MODIFY / 9% ESCALATE_OR_READONLY. Weights
applied so escalation is NOT under-taught (a safety property: the gate
must keep its ability to refuse).

In [ ]:
!pip install -q transformers peft trl datasets accelerate bitsandbytes
import torch, json
from datasets import Dataset
print(torch.cuda.device_count(), "GPUs")

## 1. Load the corpus

Upload `eval_corpus/finetune.jsonl` as a Kaggle dataset (or paste via
the input panel). Labels: `MODIFY` / `ESCALATE_OR_READONLY`.

In [ ]:
import json
rows = [json.loads(l) for l in open("/kaggle/input/janus-finetune/finetune.jsonl")]
# dedupe by prompt (rows repeat across eval modes)
seen, data = set(), []
for r in rows:
    key = r["prompt"]
    if key in seen or not r["prompt"].strip(): continue
    seen.add(key)
    data.append({"prompt": r["prompt"], "label": r["gold_intent"]})
print(f"{len(data)} unique labeled decisions")
from collections import Counter
print(Counter(d["label"] for d in data))

## 2. Build the training set

Prompt format matches the decider's consumption: instruction + repo
context (empty at decision time) → intent label. 90/10 stratified
split for the held-out accuracy read.

In [ ]:
from sklearn.model_selection import train_test_split
SYSTEM = ("You are System 1, a coding-intent decision head. Given a user "
          "request, output exactly one intent label: MODIFY or ESCALATE_OR_READONLY. "
          "Output the label only.")
def fmt(d): return {"messages": [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": d["prompt"]},
    {"role": "assistant", "content": d["label"]}]}
train, test = train_test_split(data, test_size=0.1, stratify=[d["label"] for d in data], random_state=7)
train_ds = Dataset.from_list([fmt(d) for d in train])
test_ds  = Dataset.from_list([fmt(d) for d in test])
print(f"train {len(train_ds)} / test {len(test_ds)}")

## 3. Model + LoRA

Qwen/Qwen3-0.6B (same family as the deployed decider). LoRA keeps the
adapter small enough to merge back and re-quantize for Ollama.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
MODEL = "Qwen/Qwen3-0.6B"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map="auto")
lora = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj","k_proj","v_proj","o_proj"], lora_dropout=0.05, task_type="CAUSAL_LM")
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 4. Train (with class weighting via oversampling)

Oversample ESCALATE_OR_READONLY to ~35% of the training mix so the
gate keeps its refusal ability — the GA-GRPO guidance-weight applied
at the data level, where it belongs for an SFT head.

In [ ]:
from trl import SFTTrainer, SFTConfig
esc = [d for d in train if d["label"] == "ESCALATE_OR_READONLY"]
mod = [d for d in train if d["label"] == "MODIFY"]
# oversample escalation ~4x (294 → ~1180) so it's ~35% of the mix
balanced = train + esc * 4
import random; random.seed(7); random.shuffle(balanced)
train_ds = Dataset.from_list([fmt(d) for d in balanced])
cfg = SFTConfig(output_dir="/kaggle/working/janus-s1", num_train_epochs=3,
    per_device_train_batch_size=8, gradient_accumulation_steps=2,
    learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.05,
    logging_steps=20, bf16=True, report_to="none", max_length=512)
trainer = SFTTrainer(model=model, args=cfg, train_dataset=train_ds, processing_class=tok)
trainer.train()

## 5. Held-out accuracy — per class (the number that matters)

Overall accuracy hides escalation recall. Report both: MODIFY accuracy
and ESCALATE recall. The gate's safety lives in the recall number.

In [ ]:
model.eval()
correct = {l: 0 for l in ("MODIFY","ESCALATE_OR_READONLY")}
total   = {l: 0 for l in ("MODIFY","ESCALATE_OR_READONLY")}
for d in test:
    msgs = fmt(d)["messages"][:2]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
    out = model.generate(ids, max_new_tokens=8, do_sample=False, pad_token_id=tok.eos_token_id)
    pred = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()
    for l in correct:
        if l in d["label"]: total[l] += 1; correct[l] += pred.startswith(l.split("_")[0])
for l in correct:
    print(f"{l}: {correct[l]}/{total[l]} = {correct[l]/max(total[l],1):.2%}")
print(f"overall: {sum(correct.values())}/{sum(total.values())} = {sum(correct.values())/max(sum(total.values()),1):.2%}")

## 6. Export the adapter → merged model → GGUF for Ollama

In [ ]:
merged = model.merge_and_unload()
merged.save_pretrained("/kaggle/working/janus-s1-merged")
tok.save_pretrained("/kaggle/working/janus-s1-merged")
# then locally: convert to GGUF (llama.cpp convert_hf_to_gguf.py) and
# create the Ollama model. Deployment is a fresh calibration fit — the
# fine-tuned head changes the noul distribution, so T must be refit.